In [3]:
import pandas as pd
import numpy as np


data = pd.read_csv("train_IPL.csv", low_memory=False)

FileNotFoundError: [Errno 2] No such file or directory: 'train_IPL.csv'

In [ ]:
match_data = data.groupby("Match ID").agg(
    venue=("Venue", "first"),
    bat_first=("Bat First", "first"),
    bat_second=("Bat Second", "first"),
    toss_winner=("toss_winner", "first"),
    toss_decision=("toss_decision", "first"),
    city=("city", "first"),
    result_type=("result_type", "first"),
    match_won_by=("match_won_by", "first")
).reset_index()

match_data = match_data[match_data['result_type'].isna()].copy()
print("Clean matches:", match_data.shape)

Clean matches: (1124, 9)


In [ ]:
innings_end = data.groupby(["Match ID", "Innings"]).last().reset_index()

inn1 = innings_end[innings_end['Innings'] == 1][['Match ID', 'Innings Runs', 'Innings Wickets']].copy()
inn2 = innings_end[innings_end['Innings'] == 2][['Match ID', 'Innings Runs', 'Innings Wickets']].copy()

inn1.columns = ['Match ID', 'inn1_runs', 'inn1_wickets']
inn2.columns = ['Match ID', 'inn2_runs', 'inn2_wickets']

match_data = match_data.merge(inn1, on='Match ID', how='left')
match_data = match_data.merge(inn2, on='Match ID', how='left')

In [ ]:
def get_label(row):
    won_by = row['match_won_by']
    bat_first = row['bat_first']
    if won_by == bat_first:
        margin = row['inn1_runs'] - row['inn2_runs']
        return 'A_small' if margin <= 20 else 'A_big'
    else:
        wickets_remaining = 10 - row['inn2_wickets']
        return 'B_small' if wickets_remaining <= 5 else 'B_big'

match_data['label'] = match_data.apply(get_label, axis=1)
print(match_data['label'].value_counts())

label
B_big      401
A_big      273
A_small    240
B_small    210
Name: count, dtype: int64


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

features = ['venue', 'bat_first', 'bat_second', 'toss_winner', 'toss_decision', 'city']

encoders = {}
for col in features:
    le = LabelEncoder()
    match_data[col] = le.fit_transform(match_data[col].astype(str))
    encoders[col] = le

le_label = LabelEncoder()
match_data['label_encoded'] = le_label.fit_transform(match_data['label'])

X = match_data[features]
y = match_data['label_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

print("Classes:", le_label.classes_)
print("Train score:", model.score(X_train, y_train))
print("Test score:", model.score(X_test, y_test))

Classes: ['A_big' 'A_small' 'B_big' 'B_small']
Train score: 0.3581757508342603
Test score: 0.3511111111111111


In [ ]:
public_lb = pd.read_csv('public_lb_matches.csv')
schedule = pd.read_csv('schedule.csv')
sample_sub = pd.read_csv('sample_submission.csv')

A_small_prior = 0.213523
A_big_prior = 0.242883
B_small_prior = 0.186833
B_big_prior = 0.356761

def safe_encode(col, value):
    if value in encoders[col].classes_:
        return encoders[col].transform([value])[0]
    return 0

all_matches = pd.concat([public_lb, schedule]).reset_index(drop=True)
all_matches['match_id'] = all_matches['match_id'].astype(str)

submission = sample_sub.copy()
submission['match_id'] = submission['match_id'].astype(str)
submission['A_small'] = submission['A_small'].astype(float)
submission['A_big'] = submission['A_big'].astype(float)
submission['B_small'] = submission['B_small'].astype(float)
submission['B_big'] = submission['B_big'].astype(float)

for idx, row in all_matches.iterrows():
    match_id = str(row['match_id'])
    raw_tw = row.get('toss_winner', row['team_a'])
    raw_td = row.get('toss_decision', 'bat')

    v  = safe_encode('venue', row['venue'])
    b1 = safe_encode('bat_first', row['team_a'])
    b2 = safe_encode('bat_second', row['team_b'])
    tw = safe_encode('toss_winner', raw_tw)
    td = safe_encode('toss_decision', raw_td)
    c  = safe_encode('city', row['city'])

    lr_probs = model.predict_proba([[v, b1, b2, tw, td, c]])[0]

    blended = {
        'A_big':   0.7 * A_big_prior   + 0.3 * lr_probs[0],
        'A_small': 0.7 * A_small_prior + 0.3 * lr_probs[1],
        'B_big':   0.7 * B_big_prior   + 0.3 * lr_probs[2],
        'B_small': 0.7 * B_small_prior + 0.3 * lr_probs[3]
        }

    sub_idx = submission[submission['match_id'] == match_id].index
    for col_name, prob in blended.items():
        submission.loc[sub_idx, col_name] = prob

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist

In [ ]:
submission.to_csv('submission_final.csv', index=False)
print("Done")
print(submission[['A_small','A_big','B_small','B_big']].sum(axis=1).unique())

Done
[1. 1. 1.]


In [ ]:
prior = match_data['label'].value_counts(normalize=True)
print(prior)

label
B_big      0.356762
A_big      0.242883
A_small    0.213523
B_small    0.186833
Name: proportion, dtype: float64


In [ ]:
A_small = 0.213523
A_big = 0.242883
B_small = 0.186833
B_big = 0.356762

submission = sample_sub.copy()
submission['A_small'] = A_small
submission['A_big'] = A_big
submission['B_small'] = B_small
submission['B_big'] = B_big

print(submission.head())
print("Row sums:", submission[['A_small','A_big','B_small','B_big']].sum(axis=1).unique())

  match_id   A_small     A_big   B_small     B_big
0  1473488  0.213523  0.242883  0.186833  0.356762
1  1473489  0.213523  0.242883  0.186833  0.356762
2  1473490  0.213523  0.242883  0.186833  0.356762
3  1473491  0.213523  0.242883  0.186833  0.356762
4  1473492  0.213523  0.242883  0.186833  0.356762
Row sums: [1.000001]


In [ ]:
A_small = 0.213523
A_big = 0.242883
B_small = 0.186833
B_big = 0.356761  # adjusted to make sum exactly 1.0

submission = sample_sub.copy()
submission['A_small'] = A_small
submission['A_big'] = A_big
submission['B_small'] = B_small
submission['B_big'] = B_big

print("Row sums:", submission[['A_small','A_big','B_small','B_big']].sum(axis=1).unique())

Row sums: [1.]


In [ ]:
submission.to_csv('submission.csv', index=False)
print("Done")
print(submission.shape)

Done
(53, 5)
